# Olist E-Commerce Analysis
## Notebook 1 — Data Overview

Initial exploration of the Olist Brazilian E-Commerce dataset.
Dataset covers September 2016 – October 2018, containing 99,441 orders across 9 tables.

## Setup
Loading libraries and pointing to the raw data directory.

In [7]:
import pandas as pd
import os

data_path = "../data/raw/"

# List all files
files = os.listdir(data_path)
print(files)

['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


The dataset contains 9 CSV files covering orders, customers, products, payments, reviews, sellers and geolocation.

## Dataset Overview
Checking the dimensions of each table to understand the scale of the data.

In [8]:
for file in files:
    df = pd.read_csv(data_path + file)
    print(f"{file}: {df.shape}")

olist_customers_dataset.csv: (99441, 5)
olist_geolocation_dataset.csv: (1000163, 5)
olist_orders_dataset.csv: (99441, 8)
olist_order_items_dataset.csv: (112650, 7)
olist_order_payments_dataset.csv: (103886, 5)
olist_order_reviews_dataset.csv: (99224, 7)
olist_products_dataset.csv: (32951, 9)
olist_sellers_dataset.csv: (3095, 4)
product_category_name_translation.csv: (71, 2)


**Key observations:**
- `order_items` has more rows than `orders` (112,650 vs 99,441) — some orders contain multiple items
- `geolocation` is by far the largest table (1M+ rows) — will use carefully
- `sellers` is surprisingly small (3,095) relative to the number of orders

## Orders Table Deep Dive
The orders table is the backbone of the dataset — every other table connects to it. Inspecting it first.

In [9]:
orders = pd.read_csv(data_path + "olist_orders_dataset.csv")
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### Checking data types and null counts

In [10]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


**Key observations:**
- All timestamp columns are stored as `object` (string) — will need conversion to `datetime` in cleaning
- `order_delivered_customer_date` has 2,965 nulls — orders that never arrived (cancelled, lost, in transit)
- `order_approved_at` has 160 nulls — likely cancelled orders that were never approved

### Checking order status distribution

In [11]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

**Key observations:**
- 96,478 orders (97%) were successfully delivered — healthy platform performance
- 625 cancelled and 609 unavailable orders worth investigating
- Analysis will be filtered to `delivered` orders only to ensure accurate delivery time calculations

### Checking the time range of the dataset

In [12]:
orders['order_purchase_timestamp'].min(), orders['order_purchase_timestamp'].max()

('2016-09-04 21:15:19', '2018-10-17 17:30:18')

**Dataset covers September 2016 – October 2018 — approximately 2 years of transactions.**
All analysis reflects this specific period of Olist's early growth phase.